# 01 — Exploratory Data Analysis

Goal: understand the data, verify quality, and save `clean.parquet` for the model notebook.

**Input:** `data/raw/Sample Media Spend Data.csv`  
**Output:** `data/processed/clean.parquet`, `data/processed/eda_stats.json`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RAW  = Path('../data/raw')
PROC = Path('../data/processed')
PROC.mkdir(exist_ok=True)

# Known schema — do not rename these
DATE_COL    = 'date'
DIV_COL     = 'Division'
TARGET_COL  = 'Sales'
CHANNEL_COLS = [
    'Paid_Views',
    'Google_Impressions',
    'Email_Impressions',
    'Facebook_Impressions',
    'Affiliate_Impressions',
]
CONTROL_COLS = ['Organic_Views']
# Overall_Views is excluded — it is Paid + Organic (collinear)

print('Setup complete.')

## 1. Load & parse

In [ ]:
raw = pd.read_csv(RAW / 'Sample Media Spend Data.csv')
raw['date'] = pd.to_datetime(raw['Calendar_Week'])
raw = raw.drop(columns=['Calendar_Week', 'Overall_Views'])  # drop collinear column
raw = raw.sort_values([DIV_COL, DATE_COL]).reset_index(drop=True)

print(raw.shape)
print('Date range:', raw['date'].min(), '→', raw['date'].max())
print('Divisions :', sorted(raw[DIV_COL].unique()))
raw.head()

## 2. Missing values & dtypes

In [ ]:
print('Dtypes:\n', raw.dtypes)
print('\nMissing values:')
print(raw.isnull().sum())
raw.describe()

## 3. Weeks per division — check panel is balanced

In [ ]:
weeks_per_div = raw.groupby(DIV_COL)['date'].nunique().sort_values()
print(weeks_per_div)
print('\nAll divisions have equal weeks?', weeks_per_div.nunique() == 1)

## 4. Sales time series — all divisions (small multiples)

In [ ]:
divisions = sorted(raw[DIV_COL].unique())
n = len(divisions)
ncols = 6
nrows = -(-n // ncols)  # ceiling division

fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 3), sharex=True)
for ax, div in zip(axes.flat, divisions):
    sub = raw[raw[DIV_COL] == div]
    ax.plot(sub['date'], sub[TARGET_COL], linewidth=1)
    ax.set_title(f'Division {div}', fontsize=9)
    ax.tick_params(labelsize=7)

for ax in axes.flat[n:]:
    ax.set_visible(False)

fig.suptitle('Weekly Sales per Division', fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(PROC / 'fig_sales_small_multiples.png', bbox_inches='tight')

## 5. Channel spend distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
all_cols = CHANNEL_COLS + CONTROL_COLS
for ax, col in zip(axes.flat, all_cols):
    ax.hist(raw[col].dropna(), bins=50, edgecolor='none')
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')

for ax in axes.flat[len(all_cols):]:
    ax.set_visible(False)

plt.suptitle('Channel Distributions (all divisions combined)', fontsize=12)
plt.tight_layout()
fig.savefig(PROC / 'fig_channel_distributions.png', bbox_inches='tight')

## 6. Zero-spend weeks per channel — important for adstock priors

In [ ]:
zero_pct = (raw[CHANNEL_COLS] == 0).mean() * 100
print('Percentage of zero-spend weeks per channel:')
print(zero_pct.to_string())

## 7. Correlation heatmap: channels + controls vs Sales

In [ ]:
corr_cols = CHANNEL_COLS + CONTROL_COLS + [TARGET_COL]
corr = raw[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Pearson Correlation (all divisions pooled)')
plt.tight_layout()
fig.savefig(PROC / 'fig_correlation_heatmap.png', bbox_inches='tight')

## 8. Spend vs Sales scatter per channel

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flat, CHANNEL_COLS + CONTROL_COLS):
    ax.scatter(raw[col], raw[TARGET_COL], alpha=0.15, s=10)
    ax.set_xlabel(col)
    ax.set_ylabel(TARGET_COL)
    ax.set_title(f'{col} vs {TARGET_COL}')

for ax in axes.flat[len(CHANNEL_COLS + CONTROL_COLS):]:
    ax.set_visible(False)

plt.suptitle('Channel Spend vs Sales', fontsize=12)
plt.tight_layout()
fig.savefig(PROC / 'fig_scatter_spend_vs_sales.png', bbox_inches='tight')

## 9. Rolling mean — stationarity check on Sales

In [ ]:
# Aggregate across all divisions to see the overall trend
agg = raw.groupby('date')[TARGET_COL].sum().reset_index()
agg['rolling_12w'] = agg[TARGET_COL].rolling(12, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(agg['date'], agg[TARGET_COL], alpha=0.4, label='Weekly total')
ax.plot(agg['date'], agg['rolling_12w'], linewidth=2, label='12-week rolling mean')
ax.set_title('Total Sales — All Divisions')
ax.set_xlabel('Date')
ax.set_ylabel('Sales')
ax.legend()
plt.tight_layout()
fig.savefig(PROC / 'fig_sales_rolling.png', bbox_inches='tight')

## 10. Summary stats — save for app

In [ ]:
stats = {
    'n_rows': int(len(raw)),
    'n_divisions': int(raw[DIV_COL].nunique()),
    'date_min': str(raw['date'].min().date()),
    'date_max': str(raw['date'].max().date()),
    'channel_cols': CHANNEL_COLS,
    'control_cols': CONTROL_COLS,
    'target_col': TARGET_COL,
    'channel_means': raw[CHANNEL_COLS].mean().round(1).to_dict(),
    'sales_mean': float(raw[TARGET_COL].mean().round(1)),
    'sales_std': float(raw[TARGET_COL].std().round(1)),
}

with open(PROC / 'eda_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print(json.dumps(stats, indent=2))

## 11. Save clean parquet

In [ ]:
raw.to_parquet(PROC / 'clean.parquet', index=False)
print(f'Saved {len(raw):,} rows → data/processed/clean.parquet')
print('Columns:', list(raw.columns))